# 03-1. 출시 후 LLM 분류 리뷰 전처리 및 결과 통합

이 코드는 `03_run_llm_postlaunch_analysis_final.ipynb`에서 생성한 LLM 분류 결과를 읽어, 후속 단계인 **LLM 기반 패치·운영 전략 제안**에 사용할 근거 데이터를 만든다.

흐름은 출시 전 분석의 `03-1`과 맞춘다.

```text
03. 리뷰 LLM 분류
→ 03-1. 분류된 리뷰 전처리 및 게임별 결과 통합
→ 03-2. LLM 기반 패치·운영 전략 제안
```

주의할 점은 다음과 같다.

- 이 코드에서는 LLM을 호출하지 않는다.
- LLM이 분류한 감정, 이슈 태그, urgency 후보를 그대로 최종 우선 검토 수준로 사용하지 않는다.
- 패치·운영 우선 검토 수준은 **이슈 반복 수, 부정·혼합 리뷰 수, Steam 비추천 리뷰 수, 최근 리뷰 반복 여부, 짧은 플레이타임 부정 반응**을 기준으로 규칙 기반 계산한다.
- `High urgency`는 03번 리뷰 분류 단계에서 LLM이 판단한 시급도 후보이므로, **우선 검토 수준 계산에는 직접 사용하지 않고 보조 설명 지표로만 유지한다.**
- 03-2에서는 이 코드에서 계산한 `action_group_hint`, `rule_priority_hint`, `priority_rule_detail`, `priority_reason`을 고정 근거로 사용한다.


# 1. 기본 설정

In [1]:
# ============================================================
# 기본 라이브러리
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


In [2]:
# ============================================================
# 프로젝트 경로 설정
# ============================================================
ROOT = Path.cwd()

if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# ============================================================
# 출시 후 분석 대상 게임 설정
# ============================================================
# 여러 게임을 한 번에 실행하려면 RUN_MULTI_GAME_MODE=True로 두고,
# TARGET_GAME_KEYS에 실행할 게임 key를 넣으면 된다.
# 게임 1개만 실행하려면 RUN_MULTI_GAME_MODE=False로 바꾸고 TARGET_GAME_KEY만 변경한다.

RUN_MULTI_GAME_MODE = True
RUN_CHECK_CELLS = True

POSTLAUNCH_TARGET_GAMES = {
    "heroes_of_hammerwatch_2": {
        "appid": 619820,
        "game_name": "Heroes of Hammerwatch II",
        "game_slug": "heroes_of_hammerwatch_2",
    },
    "necrosmith_2": {
        "appid": 2277320,
        "game_name": "Necrosmith 2",
        "game_slug": "necrosmith_2",
    },
    "children_of_the_sun": {
        "appid": 1309950,
        "game_name": "Children of the Sun",
        "game_slug": "children_of_the_sun",
    },
    "laundry_store_simulator": {
        "appid": 3150440,
        "game_name": "Laundry Store Simulator",
        "game_slug": "laundry_store_simulator",
    },
    "endoparasitic_2": {
        "appid": 2990640,
        "game_name": "Endoparasitic 2",
        "game_slug": "endoparasitic_2",
    },
}

# 03-1번에서 한 번에 실행할 게임 목록
TARGET_GAME_KEYS = [
    "heroes_of_hammerwatch_2",
    "necrosmith_2",
    "children_of_the_sun",
    "laundry_store_simulator",
    "endoparasitic_2",
]

# 단일 실행 모드에서 사용할 게임
TARGET_GAME_KEY = TARGET_GAME_KEYS[0]

# ============================================================
# 출시 후 게임별 데이터 폴더
# ============================================================
POSTLAUNCH_OUTPUT_DIR = ROOT / "data" / "outputs" / "postlaunch"
RUNS_DIR = POSTLAUNCH_OUTPUT_DIR / "runs"


def set_target_game_context(game_key):
    """게임 key를 기준으로 03-1 전처리에 필요한 전역 경로와 설정값을 갱신한다."""
    global TARGET_GAME_KEY, TARGET_GAME, TARGET_APPID, TARGET_GAME_NAME, GAME_SLUG
    global RUN_DIR, RESULT_CSV_PATH, ISSUE_TAG_FLAT_PATH, LLM_INPUT_PATH
    global POSTLAUNCH_PREPROCESS_DIR
    global POSTLAUNCH_REVIEW_BASE_PATH, POSTLAUNCH_ISSUE_SUMMARY_PATH
    global POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH, TABLEAU_POSTLAUNCH_SOURCE_PATH

    if game_key not in POSTLAUNCH_TARGET_GAMES:
        raise KeyError(f"POSTLAUNCH_TARGET_GAMES에 없는 game_key입니다: {game_key}")

    TARGET_GAME_KEY = game_key
    TARGET_GAME = POSTLAUNCH_TARGET_GAMES[TARGET_GAME_KEY]
    TARGET_APPID = TARGET_GAME["appid"]
    TARGET_GAME_NAME = TARGET_GAME["game_name"]
    GAME_SLUG = TARGET_GAME["game_slug"]

    RUN_DIR = RUNS_DIR / GAME_SLUG
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    RESULT_CSV_PATH = RUN_DIR / "llm_review_analysis_result.csv"
    ISSUE_TAG_FLAT_PATH = RUN_DIR / "llm_issue_tags_flat.csv"
    LLM_INPUT_PATH = RUN_DIR / "llm_input_reviews.csv"

    POSTLAUNCH_PREPROCESS_DIR = RUN_DIR / "postlaunch_preprocess_data"
    POSTLAUNCH_PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)

    POSTLAUNCH_REVIEW_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_review_base.csv"
    POSTLAUNCH_ISSUE_SUMMARY_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_issue_summary.csv"
    POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_patch_ops_evidence_base.csv"
    TABLEAU_POSTLAUNCH_SOURCE_PATH = POSTLAUNCH_PREPROCESS_DIR / "tableau_postlaunch_patch_ops_source.csv"

    return {
        "game_key": TARGET_GAME_KEY,
        "appid": TARGET_APPID,
        "game_name": TARGET_GAME_NAME,
        "run_dir": RUN_DIR,
        "result_csv_path": RESULT_CSV_PATH,
        "issue_tag_flat_path": ISSUE_TAG_FLAT_PATH,
        "preprocess_dir": POSTLAUNCH_PREPROCESS_DIR,
    }


set_target_game_context(TARGET_GAME_KEY)

print("ROOT:", ROOT)
print("RUN_MULTI_GAME_MODE:", RUN_MULTI_GAME_MODE)
print("실행 대상 게임 key 목록:", TARGET_GAME_KEYS)
print("현재 미리보기 대상:", TARGET_GAME_NAME)
print("TARGET_APPID:", TARGET_APPID)
print("GAME_SLUG:", GAME_SLUG)
print("RUN_DIR:", RUN_DIR)
print("03번 LLM 결과 CSV:", RESULT_CSV_PATH)
print("03번 이슈 태그 CSV:", ISSUE_TAG_FLAT_PATH)
print("03번 LLM 입력 CSV:", LLM_INPUT_PATH)
print("03-1 전처리 저장 폴더:", POSTLAUNCH_PREPROCESS_DIR)
print("RESULT_CSV_PATH exists:", RESULT_CSV_PATH.exists())
print("ISSUE_TAG_FLAT_PATH exists:", ISSUE_TAG_FLAT_PATH.exists())
print("LLM_INPUT_PATH exists:", LLM_INPUT_PATH.exists())


ROOT: c:\Users\joon5\Documents\github\steam-indie-game-analysis
RUN_MULTI_GAME_MODE: True
실행 대상 게임 key 목록: ['heroes_of_hammerwatch_2', 'necrosmith_2', 'children_of_the_sun', 'laundry_store_simulator', 'endoparasitic_2']
현재 미리보기 대상: Heroes of Hammerwatch II
TARGET_APPID: 619820
GAME_SLUG: heroes_of_hammerwatch_2
RUN_DIR: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2
03번 LLM 결과 CSV: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\llm_review_analysis_result.csv
03번 이슈 태그 CSV: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\llm_issue_tags_flat.csv
03번 LLM 입력 CSV: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\llm_input_reviews.csv
03-1 전처리 저장 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_

# 2. 여러 게임 한 번에 전처리 실행

`TARGET_GAME_KEYS`에 들어 있는 게임을 순서대로 전처리한다.


In [3]:
def process_postlaunch_preprocess_for_game(game_key):
    """03-1번 전체 전처리/우선순위 근거 생성 흐름을 게임 1개 기준으로 실행한다."""
    print("\n" + "=" * 90)
    print(f"03-1번 출시 후 전처리 실행: {game_key}")
    print("=" * 90)

    set_target_game_context(game_key)

    missing_inputs = [
        path for path in [RESULT_CSV_PATH, ISSUE_TAG_FLAT_PATH]
        if not path.exists()
    ]
    if missing_inputs:
        raise FileNotFoundError(
            "03번 LLM 산출물이 없습니다. 먼저 03번을 실행하세요: "
            + ", ".join(str(path) for path in missing_inputs)
        )

    # ============================================================
    # 03번 LLM 분류 결과 불러오기
    # ============================================================

    result_df = pd.read_csv(RESULT_CSV_PATH, dtype={"recommendationid": "string"})
    issue_df = pd.read_csv(ISSUE_TAG_FLAT_PATH, dtype={"recommendationid": "string"})

    print("리뷰 단위 LLM 결과:", result_df.shape)
    print("이슈 태그 단위 결과:", issue_df.shape)

    if RUN_CHECK_CELLS:
        display(result_df.head())
    if RUN_CHECK_CELLS:
        display(issue_df.head())


    # ============================================================
    # 필수 컬럼 확인
    # ============================================================

    required_result_cols = [
        "recommendationid", "appid", "game_name",
        "review_datetime", "release_date", "days_from_release", "release_period",
        "steam_label_text", "playtime_at_review_hours",
        "llm_sentiment", "llm_primary_issue", "llm_urgency_candidate",
        "llm_review_summary", "llm_suggested_action",
    ]

    required_issue_cols = [
        "recommendationid", "appid", "game_name",
        "steam_label_text", "llm_sentiment", "llm_urgency_candidate",
        "release_period", "playtime_at_review_hours",
        "llm_issue_category", "issue_name_kor", "llm_issue_sentiment", "llm_issue_evidence",
    ]

    missing_result_cols = [col for col in required_result_cols if col not in result_df.columns]
    missing_issue_cols = [col for col in required_issue_cols if col not in issue_df.columns]

    if missing_result_cols:
        raise ValueError(f"리뷰 결과 파일에 필요한 컬럼이 없습니다: {missing_result_cols}")

    if missing_issue_cols:
        raise ValueError(f"이슈 태그 파일에 필요한 컬럼이 없습니다: {missing_issue_cols}")

    print("필수 컬럼 확인 완료")


    # ============================================================
    # 공통 함수
    # ============================================================

    def safe_rate(numerator, denominator):
        """0으로 나누는 경우를 방지한 비율 계산 함수."""
        if denominator is None or pd.isna(denominator) or denominator == 0:
            return 0.0
        return round(float(numerator) / float(denominator), 4)


    def classify_playtime_stage(hours):
        """리뷰 작성 시점 플레이타임을 출시 후 운영 해석용 구간으로 분류한다."""
        if pd.isna(hours):
            return "unknown"
        if hours < 1:
            return "0-1h"
        if hours < 5:
            return "1-5h"
        if hours < 20:
            return "5-20h"
        if hours < 50:
            return "20-50h"
        return "50h+"


    def classify_recency_group(review_dt, max_dt):
        """분석 데이터 내 최신 리뷰일 기준 최근성 구간을 만든다."""
        if pd.isna(review_dt) or pd.isna(max_dt):
            return "unknown"

        diff_days = (max_dt - review_dt).days

        if diff_days <= 30:
            return "last_30d"
        if diff_days <= 60:
            return "31-60d"
        if diff_days <= 90:
            return "61-90d"
        return "older_90d"


    def normalize_text_value(x):
        """결측/공백 텍스트를 안정적으로 처리한다."""
        if pd.isna(x):
            return ""
        return str(x).strip()


    # ============================================================
    # 리뷰 단위 기본 전처리
    # ============================================================

    review_base = result_df.copy()

    # 날짜/숫자 타입 정리
    for col in ["review_datetime", "release_date"]:
        review_base[col] = pd.to_datetime(review_base[col], errors="coerce")

    numeric_cols = [
        "days_from_release", "playtime_at_review_hours", "votes_up",
        "weighted_vote_score", "sentiment_score",
    ]
    for col in numeric_cols:
        if col in review_base.columns:
            review_base[col] = pd.to_numeric(review_base[col], errors="coerce")

    # 문자열 정리
    text_cols = [
        "recommendationid", "game_name", "release_period", "steam_label_text",
        "llm_sentiment", "llm_primary_issue", "llm_urgency_candidate",
        "llm_review_summary", "llm_suggested_action",
    ]
    for col in text_cols:
        if col in review_base.columns:
            review_base[col] = review_base[col].apply(normalize_text_value)

    # 분석 기준일
    # recent_30d는 실제 오늘 기준 최근 30일이 아니라,
    # 분석 데이터 내 최신 리뷰일을 기준으로 계산한 상대적 최근성 구간이다.
    MAX_REVIEW_DATETIME = review_base["review_datetime"].max()

    # 파생 구간/플래그
    review_base["playtime_stage"] = review_base["playtime_at_review_hours"].apply(classify_playtime_stage)
    review_base["review_recency_group"] = review_base["review_datetime"].apply(lambda x: classify_recency_group(x, MAX_REVIEW_DATETIME))

    review_base["steam_negative_flag"] = review_base["steam_label_text"].eq("negative")
    review_base["steam_positive_flag"] = review_base["steam_label_text"].eq("positive")
    review_base["llm_negative_or_mixed_flag"] = review_base["llm_sentiment"].isin(["negative", "mixed"])
    review_base["llm_positive_flag"] = review_base["llm_sentiment"].eq("positive")
    review_base["high_urgency_flag"] = review_base["llm_urgency_candidate"].eq("high")
    review_base["early_playtime_flag"] = review_base["playtime_stage"].isin(["0-1h", "1-5h"])
    review_base["recent_30d_flag"] = review_base["review_recency_group"].eq("last_30d")

    # 저장 컬럼 정리
    review_base_cols = [
        "analysis_status",
        "recommendationid", "appid", "game_name",
        "review_datetime", "release_date", "days_from_release", "release_period", "review_recency_group",
        "steam_label_text", "steam_positive_flag", "steam_negative_flag",
        "playtime_at_review_hours", "playtime_stage", "early_playtime_flag", "recent_30d_flag",
        "votes_up", "weighted_vote_score",
        "llm_sentiment", "sentiment_score", "llm_negative_or_mixed_flag", "llm_positive_flag",
        "llm_primary_issue", "llm_urgency_candidate", "high_urgency_flag",
        "llm_review_summary", "llm_suggested_action", "steam_llm_sentiment_relation",
    ]
    review_base_cols = [col for col in review_base_cols if col in review_base.columns]
    review_base = review_base[review_base_cols].copy()

    print("리뷰 단위 전처리 결과:", review_base.shape)
    print("최신 리뷰일:", MAX_REVIEW_DATETIME)
    if RUN_CHECK_CELLS:
        display(review_base.head())


    # ============================================================
    # 이슈 태그 단위 기본 전처리
    # ============================================================

    issue_base = issue_df.copy()

    # 날짜/최근성/리뷰 요약 등은 리뷰 단위 결과에서 가져온다.
    review_merge_cols = [
        "recommendationid", "review_datetime", "release_date", "days_from_release",
        "review_recency_group", "playtime_stage", "steam_negative_flag", "steam_positive_flag",
        "llm_negative_or_mixed_flag", "llm_positive_flag", "high_urgency_flag",
        "early_playtime_flag", "recent_30d_flag",
        "llm_review_summary", "llm_suggested_action", "steam_llm_sentiment_relation",
    ]
    review_merge_cols = [col for col in review_merge_cols if col in review_base.columns]

    issue_base = issue_base.merge(
        review_base[review_merge_cols].drop_duplicates("recommendationid"),
        on="recommendationid",
        how="left",
        suffixes=("", "_review"),
    )

    # 숫자 타입 정리
    for col in ["playtime_at_review_hours", "votes_up", "weighted_vote_score", "days_from_release"]:
        if col in issue_base.columns:
            issue_base[col] = pd.to_numeric(issue_base[col], errors="coerce")

    # 문자열 정리
    for col in [
        "recommendationid", "game_name", "steam_label_text", "llm_sentiment",
        "llm_primary_issue", "llm_urgency_candidate", "release_period",
        "llm_issue_category", "issue_name_kor", "llm_issue_sentiment", "llm_issue_evidence",
    ]:
        if col in issue_base.columns:
            issue_base[col] = issue_base[col].apply(normalize_text_value)

    # 이슈 태그 기준 감정 플래그
    issue_base["issue_positive_flag"] = issue_base["llm_issue_sentiment"].eq("positive")
    issue_base["issue_negative_or_mixed_flag"] = issue_base["llm_issue_sentiment"].isin(["negative", "mixed"])
    issue_base["issue_negative_flag"] = issue_base["llm_issue_sentiment"].eq("negative")
    issue_base["issue_mixed_flag"] = issue_base["llm_issue_sentiment"].eq("mixed")

    # 같은 리뷰 안에서 같은 이슈가 중복으로 들어온 경우 제거
    issue_review_base = issue_base.drop_duplicates(["recommendationid", "llm_issue_category"]).copy()

    print("이슈 태그 원본 행 수:", len(issue_base))
    print("리뷰-이슈 중복 제거 후 행 수:", len(issue_review_base))
    if RUN_CHECK_CELLS:
        display(issue_review_base.head())


    # ============================================================
    # 게임 단위 기본 요약
    # ============================================================

    def make_game_base(review_df, issue_review_df):
        rows = []

        for (appid, game_name), g in review_df.groupby(["appid", "game_name"], dropna=False):
            issue_g = issue_review_df[issue_review_df["appid"] == appid]

            review_count = g["recommendationid"].nunique()
            steam_positive_count = int(g["steam_positive_flag"].sum())
            steam_negative_count = int(g["steam_negative_flag"].sum())
            llm_positive_count = int(g["llm_positive_flag"].sum())
            llm_negative_mixed_count = int(g["llm_negative_or_mixed_flag"].sum())
            high_urgency_count = int(g["high_urgency_flag"].sum())

            rows.append({
                "appid": appid,
                "game_name": game_name,
                "review_count": review_count,
                "issue_tag_count": len(issue_g),
                "steam_positive_review_count": steam_positive_count,
                "steam_negative_review_count": steam_negative_count,
                "steam_positive_rate": safe_rate(steam_positive_count, review_count),
                "llm_positive_review_count": llm_positive_count,
                "llm_negative_mixed_review_count": llm_negative_mixed_count,
                "llm_negative_mixed_rate": safe_rate(llm_negative_mixed_count, review_count),
                "high_urgency_review_count": high_urgency_count,
                "high_urgency_rate": safe_rate(high_urgency_count, review_count),
                "first_review_datetime": g["review_datetime"].min(),
                "last_review_datetime": g["review_datetime"].max(),
                "mean_playtime_at_review_hours": round(g["playtime_at_review_hours"].mean(), 2),
                "median_playtime_at_review_hours": round(g["playtime_at_review_hours"].median(), 2),
            })

        return pd.DataFrame(rows)


    game_base = make_game_base(review_base, issue_review_base)

    print("게임 단위 요약:", game_base.shape)
    if RUN_CHECK_CELLS:
        display(game_base)


    # ============================================================
    # 대응 구분/우선 검토 힌트 함수
    # ============================================================
    # 튜터님 피드백 반영:
    # 패치·운영 우선순위는 LLM이 직접 판단하지 않는다.
    # 이 셀에서 사전에 정한 데이터 기준으로 action_group_hint와 rule_priority_hint를 계산한다.
    #
    # 수정 포인트:
    # - 기존 절대 기준만 사용하면 리뷰 수가 많은 게임에서 대부분 '상'으로 분류될 수 있다.
    # - 따라서 절대 기준으로 먼저 priority_candidate를 만들고,
    #   최종 rule_priority_hint는 이번 게임 안에서 상대적으로 먼저 봐야 할 상위 이슈만 '상'으로 제한한다.
    # - High urgency는 LLM 기반 보조 지표이므로 우선 검토 수준 계산에는 직접 사용하지 않는다.
    #
    # 핵심 기준:
    # 1. 해당 이슈가 리뷰에서 반복적으로 나타났는가
    # 2. 부정·혼합 맥락으로 자주 언급되었는가
    # 3. Steam 비추천 리뷰에서도 함께 나타났는가
    # 4. 최근 30일에도 반복되는가
    # 5. 이슈 성격상 즉시 확인이 필요한 기술/진행 방해 이슈인가

    # ------------------------------------------------------------
    # 실제 03번 LLM 결과의 llm_issue_category 값과 맞춘 이슈 그룹
    # ------------------------------------------------------------
    IMMEDIATE_ISSUES = {
        "crash",
        "save_progression",
        "save_progress",
        "bug",
        "performance",
        "optimization",
    }

    SHORT_TERM_ISSUES = {
        "gameplay_loop",
        "balance",
        "difficulty",
        "ui_ux",
        "control",
        "controls",
        "progression_grind",
    }

    LONG_TERM_ISSUES = {
        "content_volume",
        "content_amount",
        "story",
        "graphics_audio",
        "multiplayer",
        "monetization",
        "pricing",
        "price_value",
        "translation_localization",
    }

    STRENGTH_ISSUES = {"positive_praise"}
    OPS_ISSUES = {"developer_communication"}

    # ------------------------------------------------------------
    # 절대 기준 기반 우선순위 후보 기준
    # ------------------------------------------------------------
    # get_rule_priority_hint는 최종 우선순위가 아니라 priority_candidate 후보값을 만드는 데 사용한다.
    # 최종 rule_priority_hint는 apply_relative_priority에서 상대 순위 보정을 거쳐 결정한다.

    HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS = 20
    HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS = 10
    HIGH_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS = 5
    HIGH_PRIORITY_MIN_AFFECTED_REVIEWS = 50

    MID_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS = 10
    MID_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS = 5
    MID_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS = 3
    MID_PRIORITY_MIN_AFFECTED_REVIEWS = 20

    # ------------------------------------------------------------
    # 최종 '상' 개수 제한 기준
    # ------------------------------------------------------------
    # '상'은 단순히 문제가 있다는 뜻이 아니라,
    # 이번 분석 대상 게임에서 먼저 확인해야 하는 상위 이슈라는 의미로 제한한다.
    # - 최대 6개
    # - 전체 개선 대상 이슈의 30% 이내
    MAX_HIGH_ISSUES = 6
    MAX_HIGH_RATE = 0.30

    # 'other'는 원인이 명확하지 않으므로 수치가 커도 바로 '상'으로 올리지 않는다.
    # positive_praise는 강점 유지 항목이므로 개선 우선순위 '상'에서 제외한다.
    NO_HIGH_ISSUES = {
        "positive_praise",
        "other",
    }

    ACTION_GROUP_RANK = {
        "즉시 확인": 0,
        "단기 개선": 1,
        "운영 커뮤니케이션 개선": 2,
        "장기 검토": 3,
        "검토 필요": 4,
        "강점 유지": 5,
    }

    PATCH_OPS_NOTE_MAP = {
        "crash": "크래시 발생 조건과 로그를 우선 확인하고, 재현 가능한 오류부터 수정한다.",
        "save_progression": "저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다.",
        "save_progress": "저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다.",
        "bug": "반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다.",
        "performance": "프레임 저하, 로딩, 끊김 등 성능 문제를 환경별로 점검한다.",
        "optimization": "최적화 이슈가 특정 구간이나 사양에서 반복되는지 확인한다.",
        "gameplay_loop": "반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다.",
        "balance": "전투, 성장, 보상, 적 난이도의 불균형 지점을 조정한다.",
        "difficulty": "초반 진입 장벽과 후반 난이도 피로를 구분해 난이도 옵션 또는 안내를 보강한다.",
        "ui_ux": "메뉴, 인벤토리, 퀘스트 안내, 조작 안내 등 편의성 문제를 개선한다.",
        "control": "이동, 전투, 상호작용 조작의 반응성과 키 설정 편의성을 점검한다.",
        "controls": "이동, 전투, 상호작용 조작의 반응성과 키 설정 편의성을 점검한다.",
        "progression_grind": "반복 성장과 노가다 피로를 줄일 수 있는 보상/성장 속도 조정을 검토한다.",
        "content_volume": "콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.",
        "content_amount": "콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.",
        "story": "서사 전달, 퀘스트 흐름, 엔딩/분기 만족도를 장기 개선 후보로 검토한다.",
        "graphics_audio": "그래픽/사운드가 몰입을 방해하는지와 강점으로 작동하는지를 함께 확인한다.",
        "multiplayer": "멀티플레이 요구가 반복되는 경우 개발 범위와 수요를 장기 로드맵에서 검토한다.",
        "monetization": "가격, DLC, 과금 관련 불만이 반복되는지 확인하고 커뮤니케이션 방식을 점검한다.",
        "pricing": "가격 대비 만족도 불만이 반복되는지 확인하고 할인/번들/콘텐츠 가치 전달을 검토한다.",
        "price_value": "가격 대비 만족도 불만이 반복되는지 확인하고 할인/번들/콘텐츠 가치 전달을 검토한다.",
        "translation_localization": "번역/현지화 불만이 실제 이해도와 진행 경험에 영향을 주는지 검토한다.",
        "developer_communication": "패치 노트, 공지, 커뮤니티 응답 등 운영 커뮤니케이션을 점검한다.",
        "positive_praise": "긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.",
        "other": "세부 리뷰를 확인해 반복되는 하위 원인이 있는지 검토한다.",
    }


    def get_action_group_hint(row):
        """이슈 성격과 부정 맥락을 기준으로 대응 구분을 계산한다."""
        issue = row["llm_issue_category"]
        affected = int(row["affected_review_count"])
        positive = int(row["positive_review_count"])
        neg_mixed = int(row["negative_mixed_review_count"])
        steam_negative = int(row["steam_negative_review_count"])

        # 긍정 칭찬은 개선 우선순위가 아니라 유지할 강점으로 분리한다.
        if issue in STRENGTH_ISSUES and positive >= neg_mixed:
            return "강점 유지"

        # 즉시 확인은 크래시/저장/버그/성능/최적화처럼 플레이를 직접 방해할 수 있는 이슈로 제한한다.
        # High urgency만으로 즉시 확인으로 올리지 않는다.
        if issue in IMMEDIATE_ISSUES and neg_mixed >= 5 and steam_negative >= 3:
            return "즉시 확인"

        if issue in SHORT_TERM_ISSUES and neg_mixed >= 5:
            return "단기 개선"

        if issue in OPS_ISSUES and (neg_mixed >= 3 or steam_negative >= 3):
            return "운영 커뮤니케이션 개선"

        if issue in LONG_TERM_ISSUES and neg_mixed >= 5:
            return "장기 검토"

        if positive > neg_mixed and positive >= max(10, affected * 0.5):
            return "강점 유지"

        return "검토 필요"


    def get_rule_priority_hint(row):
        """절대 기준 기반 우선 검토 후보값을 계산한다.

        주의:
        - 이 함수의 결과는 최종 rule_priority_hint가 아니라 priority_candidate로 사용한다.
        - 최종 rule_priority_hint는 apply_relative_priority에서 상대 순위 보정을 거친다.
        - High urgency는 LLM 기반 보조 지표이므로 이 함수의 계산식에 넣지 않는다.
        """
        action_group = row["action_group_hint"]
        affected = int(row["affected_review_count"])
        neg_mixed = int(row["negative_mixed_review_count"])
        steam_negative = int(row["steam_negative_review_count"])
        recent_negative = int(row["recent_30d_negative_mixed_review_count"])

        if action_group == "강점 유지":
            return "하"

        # 상 후보: 부정·혼합 반복 + Steam 비추천 맥락 + 최근성 또는 큰 반복 규모가 함께 확인되는 경우
        high_by_recent = (
            neg_mixed >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS
            and steam_negative >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS
            and recent_negative >= HIGH_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS
        )

        high_by_volume = (
            affected >= HIGH_PRIORITY_MIN_AFFECTED_REVIEWS
            and neg_mixed >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS
            and steam_negative >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS
        )

        # 즉시 확인 그룹은 기술/진행 차단 가능성이 있으므로 최근성 기준이 조금 약해도 상 후보로 둔다.
        high_by_blocking_issue = (
            action_group == "즉시 확인"
            and neg_mixed >= 15
            and steam_negative >= 10
        )

        if high_by_recent or high_by_volume or high_by_blocking_issue:
            return "상"

        # 중 후보: 부정·혼합 또는 Steam 비추천 맥락이 일정 수준 확인되는 경우
        mid_by_negative = (
            neg_mixed >= MID_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS
            and steam_negative >= MID_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS
        )

        mid_by_recent = (
            recent_negative >= MID_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS
            and steam_negative >= 1
        )

        mid_by_volume = (
            affected >= MID_PRIORITY_MIN_AFFECTED_REVIEWS
            and neg_mixed >= 5
        )

        if mid_by_negative or mid_by_recent or mid_by_volume:
            return "중"

        return "하"


    def apply_relative_priority(issue_summary):
        """절대 기준 후보값을 바탕으로 최종 상/중/하를 상대적으로 보정한다.

        목적:
        - 리뷰 수가 많은 게임에서 대부분의 이슈가 '상'으로 나오는 문제를 줄인다.
        - '상'은 이번 게임에서 먼저 확인할 상위 이슈로 제한한다.
        - LLM의 High urgency는 직접 사용하지 않는다.
        """
        out = issue_summary.copy()

        # 1. 절대 기준 기반 후보값 생성
        if "priority_candidate" not in out.columns:
            out["priority_candidate"] = out.apply(get_rule_priority_hint, axis=1)

        # 2. 기본값은 하
        out["rule_priority_hint"] = "하"
        out["priority_selection_note"] = "절대 기준상 하 또는 개선 우선순위 근거가 상대적으로 약함"

        # 3. 개선 우선순위 산정 대상
        priority_target_mask = (
            out["action_group_hint"].ne("강점 유지")
            & ~out["llm_issue_category"].isin(STRENGTH_ISSUES)
        )

        target_count = int(priority_target_mask.sum())
        max_high_count = min(
            MAX_HIGH_ISSUES,
            max(1, int(np.ceil(target_count * MAX_HIGH_RATE)))
        )

        # 4. 기존 절대 기준에서 '상' 후보였던 이슈 중,
        #    기타/강점 항목은 최종 '상'에서 제외한다.
        high_candidate_mask = (
            priority_target_mask
            & out["priority_candidate"].eq("상")
            & ~out["llm_issue_category"].isin(NO_HIGH_ISSUES)
        )

        high_candidates = out[high_candidate_mask].copy()

        if len(high_candidates) > 0:
            high_candidates["action_group_rank"] = (
                high_candidates["action_group_hint"]
                .map(ACTION_GROUP_RANK)
                .fillna(9)
            )

            # 대응 긴급도가 높은 그룹을 먼저 보고,
            # 같은 그룹 안에서는 부정·혼합, Steam 비추천, 최근 반복, 전체 언급 규모가 큰 순서로 정렬한다.
            high_candidates = high_candidates.sort_values(
                [
                    "action_group_rank",
                    "negative_mixed_review_count",
                    "steam_negative_review_count",
                    "recent_30d_negative_mixed_review_count",
                    "affected_review_count",
                ],
                ascending=[True, False, False, False, False],
            )

            high_index = high_candidates.head(max_high_count).index
            out.loc[high_index, "rule_priority_hint"] = "상"
            out.loc[high_index, "priority_selection_note"] = (
                f"절대 기준 '상' 후보 중 이번 게임 내 상대 우선순위 상위 {max_high_count}개로 유지"
            )

        # 5. 절대 기준에서는 상 후보였지만 최종 상위 이슈에 들지 못한 항목은 중으로 내린다.
        #    문제 근거가 없다는 뜻이 아니라, 이번 게임에서 최우선 확인 대상은 아니라는 의미다.
        demoted_high_mask = (
            priority_target_mask
            & out["priority_candidate"].eq("상")
            & out["rule_priority_hint"].ne("상")
        )
        out.loc[demoted_high_mask, "rule_priority_hint"] = "중"
        out.loc[demoted_high_mask, "priority_selection_note"] = (
            "절대 기준상 '상' 후보였으나, 최종 '상' 개수 제한 기준에 따라 '중'으로 조정"
        )

        # 6. 절대 기준 중 후보는 중으로 유지한다.
        mid_mask = (
            priority_target_mask
            & out["priority_candidate"].eq("중")
            & out["rule_priority_hint"].ne("상")
        )
        out.loc[mid_mask, "rule_priority_hint"] = "중"
        out.loc[mid_mask, "priority_selection_note"] = "절대 기준상 '중' 후보로 분류"

        # 7. 기타 이슈는 원인이 명확하지 않으므로 최종 상으로 올리지 않는다.
        other_mask = out["llm_issue_category"].eq("other")
        out.loc[other_mask & out["rule_priority_hint"].eq("상"), "rule_priority_hint"] = "중"
        out.loc[other_mask, "priority_selection_note"] = (
            "기타 이슈는 원인이 명확하지 않아 세부 리뷰 확인 대상으로 분리"
        )

        # 8. 강점 유지 항목은 개선 우선순위가 아니므로 하로 둔다.
        strength_mask = out["action_group_hint"].eq("강점 유지")
        out.loc[strength_mask, "rule_priority_hint"] = "하"
        out.loc[strength_mask, "priority_selection_note"] = "강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리"

        return out


    def get_priority_rule_detail(row):
        """우선순위가 어떤 규칙 때문에 부여되었는지 설명용 라벨을 만든다."""
        priority = row["rule_priority_hint"]
        candidate = row.get("priority_candidate", priority)
        action_group = row["action_group_hint"]
        affected = int(row["affected_review_count"])
        neg_mixed = int(row["negative_mixed_review_count"])
        steam_negative = int(row["steam_negative_review_count"])
        recent_negative = int(row["recent_30d_negative_mixed_review_count"])

        if action_group == "강점 유지":
            return "강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리"

        if row["llm_issue_category"] == "other":
            return "기타 이슈는 원인 범주가 넓어 세부 리뷰 확인 대상으로 분리"

        if priority == "상":
            return "절대 기준 상 후보 + 게임 내 상대 우선순위 상위 이슈"

        if priority == "중" and candidate == "상":
            return "절대 기준 상 후보였으나 최종 상위 이슈 제한 기준에 따라 중으로 조정"

        if priority == "중":
            if neg_mixed >= MID_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS and steam_negative >= MID_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS:
                return "부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인"
            if recent_negative >= MID_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS:
                return "최근 30일 부정·혼합 반복이 일부 확인"
            if affected >= MID_PRIORITY_MIN_AFFECTED_REVIEWS:
                return "전체 반복 규모는 있으나 상 기준에는 미달"
            return "중 우선순위 규칙 충족"

        return "반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함"


    def build_priority_reason(row):
        """표와 LLM 입력에 넣을 근거 문장을 만든다."""
        affected = int(row["affected_review_count"])
        neg_mixed = int(row["negative_mixed_review_count"])
        steam_negative = int(row["steam_negative_review_count"])
        recent_negative = int(row["recent_30d_negative_mixed_review_count"])
        early_negative = int(row["early_playtime_negative_mixed_review_count"])
        high = int(row["high_urgency_review_count"])

        parts = [
            f"영향 리뷰 {affected}개",
            f"부정·혼합 {neg_mixed}개",
            f"Steam 비추천 맥락 {steam_negative}개",
        ]

        if recent_negative > 0:
            parts.append(f"최근 30일 부정·혼합 {recent_negative}개")

        if early_negative > 0:
            parts.append(f"초반 플레이타임 부정·혼합 {early_negative}개")

        # High urgency는 우선순위 계산값이 아니라 보조 참고 지표로만 적는다.
        if high > 0:
            parts.append(f"High urgency 후보 {high}개(보조 참고)")

        if "priority_candidate" in row.index:
            parts.append(f"절대 기준 후보 {row['priority_candidate']}")

        parts.append(f"규칙 근거: {row['priority_rule_detail']}")
        return ", ".join(parts)


    # ============================================================
    # 이슈 단위 요약 생성
    # ============================================================

    def summarize_issue_group(g):
        affected = g["recommendationid"].nunique()

        positive_ids = g.loc[g["issue_positive_flag"], "recommendationid"].nunique()
        negative_ids = g.loc[g["issue_negative_flag"], "recommendationid"].nunique()
        mixed_ids = g.loc[g["issue_mixed_flag"], "recommendationid"].nunique()
        negative_mixed_ids = g.loc[g["issue_negative_or_mixed_flag"], "recommendationid"].nunique()

        high_ids = g.loc[g["high_urgency_flag"], "recommendationid"].nunique()
        high_negative_mixed_ids = g.loc[
            g["high_urgency_flag"] & g["issue_negative_or_mixed_flag"],
            "recommendationid"
        ].nunique()

        steam_negative_ids = g.loc[g["steam_negative_flag"], "recommendationid"].nunique()

        recent_30_ids = g.loc[g["recent_30d_flag"], "recommendationid"].nunique()
        recent_30_neg_mixed_ids = g.loc[
            g["recent_30d_flag"] & g["issue_negative_or_mixed_flag"],
            "recommendationid"
        ].nunique()

        early_neg_mixed_ids = g.loc[
            g["early_playtime_flag"] & g["issue_negative_or_mixed_flag"],
            "recommendationid"
        ].nunique()

        return pd.Series({
            "appid": g["appid"].iloc[0],
            "game_name": g["game_name"].iloc[0],
            "issue_name_kor": g["issue_name_kor"].iloc[0],
            "affected_review_count": affected,
            "positive_review_count": positive_ids,
            "negative_review_count": negative_ids,
            "mixed_review_count": mixed_ids,
            "negative_mixed_review_count": negative_mixed_ids,
            "steam_negative_review_count": steam_negative_ids,
            "high_urgency_review_count": high_ids,
            "high_urgency_negative_mixed_review_count": high_negative_mixed_ids,
            "recent_30d_review_count": recent_30_ids,
            "recent_30d_negative_mixed_review_count": recent_30_neg_mixed_ids,
            "early_playtime_negative_mixed_review_count": early_neg_mixed_ids,
            "avg_playtime_at_review_hours": round(g["playtime_at_review_hours"].mean(), 2),
            "median_playtime_at_review_hours": round(g["playtime_at_review_hours"].median(), 2),
        })


    issue_summary = (
        issue_review_base
        .groupby("llm_issue_category", dropna=False)
        .apply(summarize_issue_group, include_groups=False)
        .reset_index()
    )

    issue_summary["high_urgency_rate"] = issue_summary.apply(
        lambda row: safe_rate(row["high_urgency_review_count"], row["affected_review_count"]),
        axis=1,
    )
    issue_summary["negative_mixed_rate"] = issue_summary.apply(
        lambda row: safe_rate(row["negative_mixed_review_count"], row["affected_review_count"]),
        axis=1,
    )
    issue_summary["steam_negative_rate"] = issue_summary.apply(
        lambda row: safe_rate(row["steam_negative_review_count"], row["affected_review_count"]),
        axis=1,
    )
    issue_summary["recent_30d_negative_mixed_rate"] = issue_summary.apply(
        lambda row: safe_rate(row["recent_30d_negative_mixed_review_count"], row["recent_30d_review_count"]),
        axis=1,
    )

    # ------------------------------------------------------------
    # 대응 구분과 우선 검토 수준 계산
    # ------------------------------------------------------------
    # 1. action_group_hint: 이슈 성격 기반 대응 구분
    # 2. priority_candidate: 절대 기준으로 만든 우선 검토 후보
    # 3. rule_priority_hint: 후보값을 게임 내 상대 우선순위로 보정한 최종 상/중/하
    issue_summary["action_group_hint"] = issue_summary.apply(get_action_group_hint, axis=1)
    issue_summary["priority_candidate"] = issue_summary.apply(get_rule_priority_hint, axis=1)
    issue_summary = apply_relative_priority(issue_summary)

    issue_summary["priority_rule_detail"] = issue_summary.apply(get_priority_rule_detail, axis=1)
    issue_summary["priority_reason"] = issue_summary.apply(build_priority_reason, axis=1)
    issue_summary["patch_ops_note"] = issue_summary["llm_issue_category"].map(PATCH_OPS_NOTE_MAP).fillna("세부 리뷰 확인 후 대응 방향을 검토한다.")

    # 보고서에서 보기 좋은 정렬
    priority_order = {"상": 0, "중": 1, "하": 2}
    action_order = {"즉시 확인": 0, "단기 개선": 1, "운영 커뮤니케이션 개선": 2, "장기 검토": 3, "검토 필요": 4, "강점 유지": 5}

    issue_summary["priority_order"] = issue_summary["rule_priority_hint"].map(priority_order).fillna(9)
    issue_summary["action_order"] = issue_summary["action_group_hint"].map(action_order).fillna(9)

    issue_summary = issue_summary.sort_values(
        [
            "priority_order",
            "action_order",
            "negative_mixed_review_count",
            "steam_negative_review_count",
            "recent_30d_negative_mixed_review_count",
            "affected_review_count",
        ],
        ascending=[True, True, False, False, False, False],
    ).drop(columns=["priority_order", "action_order"])

    print("이슈 단위 요약:", issue_summary.shape)
    print("최종 우선 검토 수준 분포")
    if RUN_CHECK_CELLS:
        display(issue_summary["rule_priority_hint"].value_counts().rename_axis("rule_priority_hint").reset_index(name="issue_count"))
    print("절대 기준 후보 분포")
    if RUN_CHECK_CELLS:
        display(issue_summary["priority_candidate"].value_counts().rename_axis("priority_candidate").reset_index(name="issue_count"))
    if RUN_CHECK_CELLS:
        display(issue_summary.head(20))


    # ============================================================
    # 플레이타임 구간별 이슈 요약
    # ============================================================

    playtime_order = ["0-1h", "1-5h", "5-20h", "20-50h", "50h+", "unknown"]

    playtime_issue_summary = (
        issue_review_base
        .groupby(["playtime_stage", "llm_issue_category", "issue_name_kor"], dropna=False)
        .agg(
            affected_review_count=("recommendationid", "nunique"),
            negative_mixed_review_count=("issue_negative_or_mixed_flag", "sum"),
            high_urgency_review_count=("high_urgency_flag", "sum"),
            steam_negative_review_count=("steam_negative_flag", "sum"),
        )
        .reset_index()
    )

    playtime_issue_summary["playtime_stage"] = pd.Categorical(
        playtime_issue_summary["playtime_stage"],
        categories=playtime_order,
        ordered=True,
    )
    playtime_issue_summary = playtime_issue_summary.sort_values(
        ["playtime_stage", "negative_mixed_review_count", "high_urgency_review_count"],
        ascending=[True, False, False],
    )

    print("플레이타임 구간별 이슈 요약:", playtime_issue_summary.shape)
    if RUN_CHECK_CELLS:
        display(playtime_issue_summary.head(20))


    # ============================================================
    # 최근성 구간별 이슈 요약
    # ============================================================

    recency_order = ["last_30d", "31-60d", "61-90d", "older_90d", "unknown"]

    recency_issue_summary = (
        issue_review_base
        .groupby(["review_recency_group", "llm_issue_category", "issue_name_kor"], dropna=False)
        .agg(
            affected_review_count=("recommendationid", "nunique"),
            negative_mixed_review_count=("issue_negative_or_mixed_flag", "sum"),
            high_urgency_review_count=("high_urgency_flag", "sum"),
            steam_negative_review_count=("steam_negative_flag", "sum"),
        )
        .reset_index()
    )

    recency_issue_summary["review_recency_group"] = pd.Categorical(
        recency_issue_summary["review_recency_group"],
        categories=recency_order,
        ordered=True,
    )
    recency_issue_summary = recency_issue_summary.sort_values(
        ["review_recency_group", "negative_mixed_review_count", "high_urgency_review_count"],
        ascending=[True, False, False],
    )

    print("최근성 구간별 이슈 요약:", recency_issue_summary.shape)
    if RUN_CHECK_CELLS:
        display(recency_issue_summary.head(20))


    # ============================================================
    # 03-2 LLM 입력용 근거 문장 생성 함수
    # ============================================================

    def clean_evidence_text(x, max_len=180):
        text = normalize_text_value(x)
        text = re.sub(r"\s+", " ", text)
        if len(text) > max_len:
            return text[:max_len].rstrip() + "..."
        return text


    def collect_examples(issue_category, max_examples=5):
        """이슈별 대표 리뷰 요약/근거/개선 제안을 가져온다."""
        g = issue_review_base[issue_review_base["llm_issue_category"] == issue_category].copy()

        if len(g) == 0:
            return ""

        # 부정/혼합 + 최근 리뷰 + Steam 비추천 리뷰를 우선적으로 보여준다.
        # High urgency는 보조 참고 지표이므로 대표 리뷰 정렬에서 가장 앞 기준으로 쓰지 않는다.
        g["sort_negative"] = g["issue_negative_or_mixed_flag"].astype(int)
        g["sort_steam_negative"] = g["steam_negative_flag"].astype(int)
        g["sort_recent"] = g["recent_30d_flag"].astype(int)
        g["sort_votes"] = pd.to_numeric(g.get("votes_up", 0), errors="coerce").fillna(0)

        g = g.sort_values(
            ["sort_negative", "sort_steam_negative", "sort_recent", "sort_votes"],
            ascending=[False, False, False, False],
        )

        examples = []
        used = set()

        for _, row in g.iterrows():
            rec_id = row["recommendationid"]
            if rec_id in used:
                continue
            used.add(rec_id)

            evidence = clean_evidence_text(row.get("llm_issue_evidence", ""))
            summary = clean_evidence_text(row.get("llm_review_summary", ""))
            suggested = clean_evidence_text(row.get("llm_suggested_action", ""))

            example = (
                f"- steam_label={row.get('steam_label_text', '')}, "
                f"issue_sentiment={row.get('llm_issue_sentiment', '')}, "
                f"urgency_candidate={row.get('llm_urgency_candidate', '')}, "
                f"playtime={row.get('playtime_stage', '')}, "
                f"recency={row.get('review_recency_group', '')} | "
                f"근거: {evidence} | 요약: {summary} | LLM 개선 제안 후보: {suggested}"
            )
            examples.append(example)

            if len(examples) >= max_examples:
                break

        return "\n".join(examples)


    def build_llm_evidence_text(row):
        """03-2 프롬프트에 바로 넣기 쉬운 이슈별 근거 블록을 만든다."""
        examples = collect_examples(row["llm_issue_category"], max_examples=5)

        return f"""
    [ISSUE]
    issue_category: {row['llm_issue_category']}
    issue_name_kor: {row['issue_name_kor']}
    action_group_hint: {row['action_group_hint']}
    priority_candidate: {row.get('priority_candidate', '')}  # 절대 기준 후보
    rule_priority_hint: {row['rule_priority_hint']}  # 상대 보정 후 최종 우선 검토 수준
    priority_selection_note: {row.get('priority_selection_note', '')}
    priority_rule_detail: {row['priority_rule_detail']}
    affected_review_count: {int(row['affected_review_count'])}
    negative_mixed_review_count: {int(row['negative_mixed_review_count'])}
    steam_negative_review_count: {int(row['steam_negative_review_count'])}
    recent_30d_negative_mixed_review_count: {int(row['recent_30d_negative_mixed_review_count'])}
    early_playtime_negative_mixed_review_count: {int(row['early_playtime_negative_mixed_review_count'])}
    high_urgency_review_count: {int(row['high_urgency_review_count'])}  # 보조 참고 지표
    high_urgency_rate: {row['high_urgency_rate']}  # 보조 참고 지표
    priority_reason: {row['priority_reason']}
    patch_ops_note: {row['patch_ops_note']}
    대표 근거:
    {examples}
    [/ISSUE]
    """.strip()


    # ============================================================
    # 03-2 LLM 입력용 근거 테이블 생성
    # ============================================================

    patch_ops_evidence_base = issue_summary.copy()
    patch_ops_evidence_base["llm_evidence_text"] = patch_ops_evidence_base.apply(build_llm_evidence_text, axis=1)

    # 03-2에서 너무 많은 이슈를 모두 넣지 않도록 기본 정렬 상태로 저장한다.
    # 필요하면 03-2에서 상/중 우선순위만 필터링해서 사용할 수 있다.

    print("패치·운영 전략 생성용 근거 테이블:", patch_ops_evidence_base.shape)
    if RUN_CHECK_CELLS:
        display(patch_ops_evidence_base[[
        "llm_issue_category", "issue_name_kor", "action_group_hint",
        "priority_candidate", "rule_priority_hint", "priority_selection_note",
        "priority_rule_detail", "affected_review_count", "negative_mixed_review_count",
        "steam_negative_review_count", "high_urgency_review_count",
        "recent_30d_negative_mixed_review_count", "early_playtime_negative_mixed_review_count",
        "priority_reason", "patch_ops_note",
    ]].head(20))


    # ============================================================
    # Tableau/보고서용 원천 데이터 생성
    # ============================================================

    summary_cols_for_merge = [
        "llm_issue_category", "action_group_hint", "priority_candidate", "rule_priority_hint",
        "priority_selection_note", "priority_rule_detail", "priority_reason",
        "affected_review_count", "negative_mixed_review_count", "steam_negative_review_count",
        "high_urgency_review_count", "high_urgency_rate",
        "negative_mixed_rate", "steam_negative_rate",
        "recent_30d_negative_mixed_review_count",
        "early_playtime_negative_mixed_review_count",
        "patch_ops_note",
    ]

    summary_cols_for_merge = [col for col in summary_cols_for_merge if col in issue_summary.columns]

    tableau_source = issue_review_base.merge(
        issue_summary[summary_cols_for_merge],
        on="llm_issue_category",
        how="left",
        suffixes=("", "_issue_summary"),
    )

    # Tableau에서 쓰기 좋은 컬럼만 선택
    keep_cols = [
        "appid", "game_name", "recommendationid",
        "review_datetime", "release_date", "days_from_release", "release_period", "review_recency_group",
        "steam_label_text", "llm_sentiment", "steam_llm_sentiment_relation",
        "playtime_at_review_hours", "playtime_stage", "early_playtime_flag",
        "votes_up", "weighted_vote_score",
        "llm_urgency_candidate", "high_urgency_flag",
        "llm_issue_category", "issue_name_kor", "llm_issue_sentiment",
        "issue_positive_flag", "issue_negative_or_mixed_flag",
        "action_group_hint", "priority_candidate", "rule_priority_hint",
        "priority_selection_note", "priority_rule_detail", "priority_reason",
        "affected_review_count", "negative_mixed_review_count", "steam_negative_review_count",
        "high_urgency_review_count", "high_urgency_rate", "negative_mixed_rate", "steam_negative_rate",
        "recent_30d_negative_mixed_review_count", "early_playtime_negative_mixed_review_count",
        "patch_ops_note", "llm_issue_evidence",
    ]
    keep_cols = [col for col in keep_cols if col in tableau_source.columns]
    tableau_source = tableau_source[keep_cols].copy()

    print("Tableau/보고서 원천 데이터:", tableau_source.shape)
    if RUN_CHECK_CELLS:
        display(tableau_source.head())


    # ============================================================
    # CSV 저장
    # ============================================================
    # 03-1 산출물은 03-2에서 실제로 사용할 최소 파일만 저장한다.

    review_base.to_csv(POSTLAUNCH_REVIEW_BASE_PATH, index=False, encoding="utf-8-sig")
    issue_summary.to_csv(POSTLAUNCH_ISSUE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
    patch_ops_evidence_base.to_csv(POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH, index=False, encoding="utf-8-sig")
    tableau_source.to_csv(TABLEAU_POSTLAUNCH_SOURCE_PATH, index=False, encoding="utf-8-sig")

    saved_files = pd.DataFrame([
        {"파일명": POSTLAUNCH_REVIEW_BASE_PATH.name, "행 수": len(review_base), "역할": "리뷰 1개 단위 전처리 결과"},
        {"파일명": POSTLAUNCH_ISSUE_SUMMARY_PATH.name, "행 수": len(issue_summary), "역할": "이슈별 반복성/부정·혼합/Steam 비추천/High urgency 보조 지표 요약"},
        {"파일명": POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH.name, "행 수": len(patch_ops_evidence_base), "역할": "03-2 LLM 패치·운영 전략 생성용 근거 데이터"},
        {"파일명": TABLEAU_POSTLAUNCH_SOURCE_PATH.name, "행 수": len(tableau_source), "역할": "Tableau/보고서용 원천 데이터"},
    ])

    print("저장 완료")
    print("저장 폴더:", POSTLAUNCH_PREPROCESS_DIR)
    if RUN_CHECK_CELLS:
        display(saved_files)

    # ============================================================
    # 우선순위 산정 로직 검증
    # ============================================================
    # 목적:
    # 튜터님 피드백이 코드에 실제로 반영되었는지 확인한다.
    #
    # 검증 방향:
    # 1. LLM의 High urgency가 rule_priority_hint 계산에 직접 들어가지 않았는지 확인한다.
    # 2. 최종 '상'은 반드시 rule 기반 priority_candidate == '상' 후보에서만 나오도록 확인한다.
    # 3. '즉시 확인' 이슈는 기술/진행 차단 가능성이 있는 이슈로만 제한한다.
    # 4. 리뷰 수가 많은 게임에서 '상'이 과도하게 많이 나오지 않도록 상대 보정이 적용되었는지 확인한다.
    # 5. 03-2 입력용 근거 텍스트가 비어 있지 않은지 확인한다.

    # ------------------------------------------------------------
    # 0. 검증용 기준
    # ------------------------------------------------------------
    # get_rule_priority_hint 내부의 blocking 예외 기준과 맞춘다.
    # 즉시 확인 이슈는 크래시/저장/진행/버그/성능처럼 플레이를 직접 막을 수 있으므로
    # 일반 '상' 기준보다 부정·혼합 기준을 조금 낮게 둔다.
    BLOCKING_HIGH_MIN_NEGATIVE_MIXED_REVIEWS = 15
    BLOCKING_HIGH_MIN_STEAM_NEGATIVE_REVIEWS = 10

    FORBIDDEN_PRIORITY_REFS = {
        "high_urgency_review_count",
        "high_urgency",
    }

    # ------------------------------------------------------------
    # 1. 리뷰 단위 결과는 recommendationid가 중복되면 안 된다.
    # ------------------------------------------------------------
    assert review_base["recommendationid"].is_unique, "review_base에 recommendationid 중복이 있습니다."

    # ------------------------------------------------------------
    # 2. 이슈 요약의 affected_review_count는 전체 분석 리뷰 수보다 클 수 없다.
    # ------------------------------------------------------------
    total_review_count = review_base["recommendationid"].nunique()

    affected_invalid = issue_summary[
        issue_summary["affected_review_count"] > total_review_count
    ]
    assert len(affected_invalid) == 0, affected_invalid[[
        "llm_issue_category",
        "issue_name_kor",
        "affected_review_count",
    ]]

    # ------------------------------------------------------------
    # 3. 부정·혼합 리뷰 수는 영향 리뷰 수보다 클 수 없다.
    # ------------------------------------------------------------
    negative_count_invalid = issue_summary[
        issue_summary["negative_mixed_review_count"] > issue_summary["affected_review_count"]
    ]
    assert len(negative_count_invalid) == 0, negative_count_invalid[[
        "llm_issue_category",
        "issue_name_kor",
        "affected_review_count",
        "negative_mixed_review_count",
    ]]

    # ------------------------------------------------------------
    # 4. High urgency 리뷰 수는 영향 리뷰 수보다 클 수 없다.
    # ------------------------------------------------------------
    high_urgency_count_invalid = issue_summary[
        issue_summary["high_urgency_review_count"] > issue_summary["affected_review_count"]
    ]
    assert len(high_urgency_count_invalid) == 0, high_urgency_count_invalid[[
        "llm_issue_category",
        "issue_name_kor",
        "affected_review_count",
        "high_urgency_review_count",
    ]]

    # ------------------------------------------------------------
    # 5. get_rule_priority_hint 함수 내부에서 High urgency 계열 컬럼을 직접 참조하지 않는지 확인한다.
    # ------------------------------------------------------------
    priority_func_names = {str(x) for x in get_rule_priority_hint.__code__.co_names}
    priority_func_consts = {
        str(x)
        for x in get_rule_priority_hint.__code__.co_consts
        if isinstance(x, str)
    }
    priority_func_refs = priority_func_names | priority_func_consts

    priority_forbidden_refs = [
        ref
        for ref in priority_func_refs
        if any(forbidden in ref for forbidden in FORBIDDEN_PRIORITY_REFS)
    ]

    assert len(priority_forbidden_refs) == 0, (
        "get_rule_priority_hint 계산 함수에서 High urgency 계열 참조가 발견되었습니다: "
        f"{priority_forbidden_refs}"
    )

    # ------------------------------------------------------------
    # 6. apply_relative_priority 함수 내부에서도 High urgency 계열 컬럼을 직접 참조하지 않는지 확인한다.
    # ------------------------------------------------------------
    relative_func_names = {str(x) for x in apply_relative_priority.__code__.co_names}
    relative_func_consts = {
        str(x)
        for x in apply_relative_priority.__code__.co_consts
        if isinstance(x, str)
    }
    relative_func_refs = relative_func_names | relative_func_consts

    relative_forbidden_refs = [
        ref
        for ref in relative_func_refs
        if any(forbidden in ref for forbidden in FORBIDDEN_PRIORITY_REFS)
    ]

    assert len(relative_forbidden_refs) == 0, (
        "apply_relative_priority 계산 함수에서 High urgency 계열 참조가 발견되었습니다: "
        f"{relative_forbidden_refs}"
    )

    # ------------------------------------------------------------
    # 7. priority_candidate == '상'은 명확한 rule 근거가 있어야 한다.
    # ------------------------------------------------------------
    # 일반 '상' 기준 1:
    # 부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 부정·혼합 반복
    candidate_high_by_recent = (
        (issue_summary["negative_mixed_review_count"] >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS)
        & (issue_summary["steam_negative_review_count"] >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS)
        & (issue_summary["recent_30d_negative_mixed_review_count"] >= HIGH_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS)
    )

    # 일반 '상' 기준 2:
    # 영향 리뷰 규모가 크고, 부정·혼합 + Steam 비추천 근거가 함께 있는 경우
    candidate_high_by_volume = (
        (issue_summary["affected_review_count"] >= HIGH_PRIORITY_MIN_AFFECTED_REVIEWS)
        & (issue_summary["negative_mixed_review_count"] >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS)
        & (issue_summary["steam_negative_review_count"] >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS)
    )

    # 즉시 확인 예외 기준:
    # 기술/진행 차단 가능성이 있는 이슈는 일반 기준보다 낮은 부정·혼합 기준을 허용한다.
    candidate_high_by_blocking_issue = (
        (issue_summary["action_group_hint"] == "즉시 확인")
        & (issue_summary["negative_mixed_review_count"] >= BLOCKING_HIGH_MIN_NEGATIVE_MIXED_REVIEWS)
        & (issue_summary["steam_negative_review_count"] >= BLOCKING_HIGH_MIN_STEAM_NEGATIVE_REVIEWS)
    )

    candidate_high_has_rule_basis = (
        candidate_high_by_recent
        | candidate_high_by_volume
        | candidate_high_by_blocking_issue
    )

    candidate_high_without_rule_basis = issue_summary[
        (issue_summary["priority_candidate"] == "상")
        & ~candidate_high_has_rule_basis
    ]

    assert len(candidate_high_without_rule_basis) == 0, candidate_high_without_rule_basis[[
        "llm_issue_category",
        "issue_name_kor",
        "action_group_hint",
        "affected_review_count",
        "negative_mixed_review_count",
        "steam_negative_review_count",
        "recent_30d_negative_mixed_review_count",
        "priority_candidate",
    ]]

    # ------------------------------------------------------------
    # 8. 최종 '상'은 반드시 priority_candidate == '상' 후보에서만 나와야 한다.
    # ------------------------------------------------------------
    final_high_without_candidate = issue_summary[
        (issue_summary["rule_priority_hint"] == "상")
        & (issue_summary["priority_candidate"] != "상")
    ]

    assert len(final_high_without_candidate) == 0, final_high_without_candidate[[
        "llm_issue_category",
        "issue_name_kor",
        "priority_candidate",
        "rule_priority_hint",
        "priority_selection_note",
    ]]

    # ------------------------------------------------------------
    # 9. High urgency만 많고 rule 근거가 없는데 최종 '상'으로 올라간 이슈가 없어야 한다.
    # ------------------------------------------------------------
    # 여기서는 부정·혼합 20개 미만을 무조건 금지하지 않는다.
    # save_progression 같은 즉시 확인 이슈는 blocking 기준을 만족하면 '상'이 될 수 있기 때문이다.
    high_urgency_only_high = issue_summary[
        (issue_summary["rule_priority_hint"] == "상")
        & (issue_summary["high_urgency_review_count"] >= 10)
        & ~candidate_high_has_rule_basis
    ]

    assert len(high_urgency_only_high) == 0, high_urgency_only_high[[
        "llm_issue_category",
        "issue_name_kor",
        "action_group_hint",
        "high_urgency_review_count",
        "affected_review_count",
        "negative_mixed_review_count",
        "steam_negative_review_count",
        "recent_30d_negative_mixed_review_count",
        "priority_candidate",
        "rule_priority_hint",
    ]]

    # ------------------------------------------------------------
    # 10. 즉시 확인은 기술/진행 차단 가능성이 있는 이슈 그룹으로만 제한한다.
    # ------------------------------------------------------------
    immediate_invalid = issue_summary[
        (issue_summary["action_group_hint"] == "즉시 확인")
        & (~issue_summary["llm_issue_category"].isin(IMMEDIATE_ISSUES))
    ]

    assert len(immediate_invalid) == 0, immediate_invalid[[
        "llm_issue_category",
        "issue_name_kor",
        "action_group_hint",
    ]]

    # ------------------------------------------------------------
    # 11. 강점 유지 항목은 개선 우선순위 '상'으로 들어가면 안 된다.
    # ------------------------------------------------------------
    strength_high = issue_summary[
        (issue_summary["action_group_hint"] == "강점 유지")
        & (issue_summary["rule_priority_hint"] == "상")
    ]

    assert len(strength_high) == 0, strength_high[[
        "llm_issue_category",
        "issue_name_kor",
        "action_group_hint",
        "rule_priority_hint",
    ]]

    # ------------------------------------------------------------
    # 12. 기타 이슈는 원인이 명확하지 않으므로 최종 '상'으로 들어가면 안 된다.
    # ------------------------------------------------------------
    other_high = issue_summary[
        (issue_summary["llm_issue_category"] == "other")
        & (issue_summary["rule_priority_hint"] == "상")
    ]

    assert len(other_high) == 0, other_high[[
        "llm_issue_category",
        "issue_name_kor",
        "rule_priority_hint",
        "priority_selection_note",
    ]]

    # ------------------------------------------------------------
    # 13. 최종 '상' 개수는 상대 보정 기준을 넘으면 안 된다.
    # ------------------------------------------------------------
    priority_target_count = int((
        issue_summary["action_group_hint"].ne("강점 유지")
        & ~issue_summary["llm_issue_category"].isin(STRENGTH_ISSUES)
    ).sum())

    max_high_count = min(
        MAX_HIGH_ISSUES,
        max(1, int(np.ceil(priority_target_count * MAX_HIGH_RATE)))
    )

    actual_high_count = int((issue_summary["rule_priority_hint"] == "상").sum())

    assert actual_high_count <= max_high_count, (
        f"최종 '상' 이슈 수가 제한 기준을 초과했습니다: {actual_high_count} > {max_high_count}"
    )

    # ------------------------------------------------------------
    # 14. 절대 기준 상 후보였으나 최종 상이 아닌 항목은 중으로 조정되어야 한다.
    # ------------------------------------------------------------
    # 단, 강점 유지/기타 이슈는 별도 분리 대상이므로 예외로 둔다.
    demoted_invalid = issue_summary[
        (issue_summary["priority_candidate"] == "상")
        & (issue_summary["rule_priority_hint"] == "하")
        & (issue_summary["action_group_hint"] != "강점 유지")
        & (~issue_summary["llm_issue_category"].isin(NO_HIGH_ISSUES))
    ]

    assert len(demoted_invalid) == 0, demoted_invalid[[
        "llm_issue_category",
        "issue_name_kor",
        "priority_candidate",
        "rule_priority_hint",
        "priority_selection_note",
    ]]

    # ------------------------------------------------------------
    # 15. 03-2 입력용 근거 텍스트가 비어 있지 않은지 확인한다.
    # ------------------------------------------------------------
    evidence_empty = patch_ops_evidence_base[
        patch_ops_evidence_base["llm_evidence_text"].isna()
        | patch_ops_evidence_base["llm_evidence_text"].astype(str).str.strip().eq("")
    ]

    assert len(evidence_empty) == 0, evidence_empty[[
        "llm_issue_category",
        "issue_name_kor",
        "llm_evidence_text",
    ]]

    # ------------------------------------------------------------
    # 검증 결과 출력
    # ------------------------------------------------------------
    print("검증 통과")
    print("리뷰 수:", review_base["recommendationid"].nunique())
    print("리뷰-이슈 수:", len(issue_review_base))
    print("이슈 종류 수:", issue_summary["llm_issue_category"].nunique())

    print("절대 기준 후보 분포:")
    if RUN_CHECK_CELLS:
        display(
        issue_summary["priority_candidate"]
        .value_counts()
        .rename_axis("priority_candidate")
        .reset_index(name="issue_count")
    )

    print("최종 우선 검토 수준 분포:")
    if RUN_CHECK_CELLS:
        display(
        issue_summary["rule_priority_hint"]
        .value_counts()
        .rename_axis("rule_priority_hint")
        .reset_index(name="issue_count")
    )

    print(f"최종 '상' 제한 기준: 최대 {max_high_count}개")
    print("High urgency는 rule_priority_hint 계산에는 직접 사용하지 않고, priority_reason/llm_evidence_text에서 보조 지표로만 유지한다.")

    return {
        "game_key": game_key,
        "appid": TARGET_APPID,
        "game_name": TARGET_GAME_NAME,
        "status": "success",
        "review_base_rows": len(review_base),
        "issue_summary_rows": len(issue_summary),
        "patch_ops_evidence_rows": len(patch_ops_evidence_base),
        "tableau_source_rows": len(tableau_source),
        "preprocess_dir": str(POSTLAUNCH_PREPROCESS_DIR),
        "review_base_path": str(POSTLAUNCH_REVIEW_BASE_PATH),
        "issue_summary_path": str(POSTLAUNCH_ISSUE_SUMMARY_PATH),
        "patch_ops_evidence_base_path": str(POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH),
        "tableau_source_path": str(TABLEAU_POSTLAUNCH_SOURCE_PATH),
        "error_message": "",
    }


def run_postlaunch_preprocess_multi_games(game_keys):
    """03-1번 출시 후 전처리를 여러 게임에 대해 순서대로 실행한다."""
    batch_logs = []

    for game_key in game_keys:
        try:
            log = process_postlaunch_preprocess_for_game(game_key)
        except Exception as e:
            log = {
                "game_key": game_key,
                "appid": None,
                "game_name": POSTLAUNCH_TARGET_GAMES.get(game_key, {}).get("game_name", ""),
                "status": "failed",
                "review_base_rows": None,
                "issue_summary_rows": None,
                "patch_ops_evidence_rows": None,
                "tableau_source_rows": None,
                "preprocess_dir": "",
                "review_base_path": "",
                "issue_summary_path": "",
                "patch_ops_evidence_base_path": "",
                "tableau_source_path": "",
                "error_message": str(e),
            }
            print(f"[실패] {game_key}: {e}")

        batch_logs.append(log)

    batch_log_df = pd.DataFrame(batch_logs)
    batch_log_path = RUNS_DIR / "03-1_postlaunch_preprocess_batch_log.csv"
    batch_log_df.to_csv(batch_log_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 90)
    print("03-1번 여러 게임 실행 요약")
    print("=" * 90)
    print("batch log 저장:", batch_log_path)
    display(batch_log_df)

    return batch_log_df


if RUN_MULTI_GAME_MODE:
    batch_log_df = run_postlaunch_preprocess_multi_games(TARGET_GAME_KEYS)
else:
    single_log = process_postlaunch_preprocess_for_game(TARGET_GAME_KEY)
    display(pd.DataFrame([single_log]))



03-1번 출시 후 전처리 실행: heroes_of_hammerwatch_2
리뷰 단위 LLM 결과: (1000, 20)
이슈 태그 단위 결과: (1948, 15)


,analysis_status,recommendationid,appid,game_name,review_datetime,release_date,days_from_release,release_period,steam_label_text,playtime_at_review_hours,votes_up,weighted_vote_score,llm_sentiment,sentiment_score,llm_primary_issue,llm_issue_tags,llm_urgency_candidate,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation
0,success,189880229,619820,Heroes of Hammerwatch II,2025-03-10 15:54:38,2025-01-14,55,D31-D90,negative,2.416667,3,0.510774,negative,1,save_progression,"[{""category"": ""save_progression"", ""sentiment"": ""negative"", ""evidence"": ""last 2 days I got my saved wiped out twice""}]",high,최근 2일 동안 세이브 데이터가 두 번이나 삭제되어 게임을 플레이할 수 없음.,"세이브 데이터 손실 원인 파악 및 클라우드 저장 시스템 점검, 데이터 복구 방안 검토.",exact_match
1,success,213786980,619820,Heroes of Hammerwatch II,2025-12-21 03:28:01,2025-01-14,341,D181+,positive,18.450000,12,0.655379,positive,5,positive_praise,"[{""category"": ""progression_grind"", ""sentiment"": ""neutral"", ""evidence"": ""This game can appear to be very grindy at fi...",low,"초반에는 성장이 더뎌 지루할 수 있으나, 캐릭터와 마을을 육성할수록 매우 재미있어지는 게임. 싱글 플레이 경험에 만족함.","초반 진입 장벽을 낮추기 위한 튜토리얼 강화 또는 가이드 제공, 현재의 성장 루프 유지 및 강화.",exact_match
2,success,215844634,619820,Heroes of Hammerwatch II,2026-01-12 19:36:41,2025-01-14,363,D181+,positive,30.783333,0,0.500000,mixed,3,save_progression,"[{""category"": ""save_progression"", ""sentiment"": ""negative"", ""evidence"": ""switched to linux and I lost all my data in ...",high,"게임 자체는 매우 재미있으나, 리눅스로 OS를 변경하는 과정에서 세이브 데이터가 모두 소실되어 플레이를 중단함.",OS 변경 또는 PC 교체 시 세이브 데이터 호환성 및 클라우드 동기화 문제 조사 및 수정.,partial_match
3,success,185667334,619820,Heroes of Hammerwatch II,2025-01-15 20:08:55,2025-01-14,1,D0-D30,positive,5.750000,2,0.505464,positive,5,positive_praise,"[{""category"": ""positive_praise"", ""sentiment"": ""positive"", ""evidence"": ""새로운 그래픽과 유연한 클래스 시스템, 훌륭한 협동 게임 경험""}, {""categ...",low,"그래픽과 클래스 시스템이 개선되었으며, 가격 대비 플레이 타임이 훌륭한 협동 게임으로 평가함.","현재의 그래픽 스타일과 유연한 클래스 시스템을 유지하고, 협동 플레이의 재미를 강화하는 업데이트를 지속할 것.",exact_match
4,success,186334804,619820,Heroes of Hammerwatch II,2025-01-25 00:44:23,2025-01-14,11,D0-D30,negative,3.200000,2,0.536785,negative,1,content_volume,"[{""category"": ""gameplay_loop"", ""sentiment"": ""negative"", ""evidence"": ""매우 반복적이고 게임플레이 다양성이 부족함""}, {""category"": ""conten...",high,"게임플레이가 반복적이고 콘텐츠가 부족하며, 현대적인 경쟁작들에 비해 전반적으로 수준이 낮다고 평가함.","적 종류 추가, 스킬 트리 확장, 게임플레이 다양성을 높일 수 있는 콘텐츠 업데이트 고려 필요.",exact_match


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,llm_primary_issue,llm_urgency_candidate,release_period,playtime_at_review_hours,votes_up,weighted_vote_score,llm_issue_category,issue_name_kor,llm_issue_sentiment,llm_issue_evidence
0,189880229,619820,Heroes of Hammerwatch II,negative,negative,save_progression,high,D31-D90,2.416667,3,0.510774,save_progression,저장/진행,negative,last 2 days I got my saved wiped out twice
1,213786980,619820,Heroes of Hammerwatch II,positive,positive,positive_praise,low,D181+,18.450000,12,0.655379,progression_grind,성장/반복 노가다,neutral,This game can appear to be very grindy at first
2,213786980,619820,Heroes of Hammerwatch II,positive,positive,positive_praise,low,D181+,18.450000,12,0.655379,positive_praise,긍정 칭찬,positive,I find myself keep coming back and having a great time
3,215844634,619820,Heroes of Hammerwatch II,positive,mixed,save_progression,high,D181+,30.783333,0,0.500000,save_progression,저장/진행,negative,switched to linux and I lost all my data in the game
4,215844634,619820,Heroes of Hammerwatch II,positive,mixed,save_progression,high,D181+,30.783333,0,0.500000,positive_praise,긍정 칭찬,positive,I played the game for 30 hours and it was really fun!


필수 컬럼 확인 완료
리뷰 단위 전처리 결과: (1000, 28)
최신 리뷰일: 2026-04-26 14:59:21


,analysis_status,recommendationid,appid,game_name,review_datetime,release_date,days_from_release,release_period,review_recency_group,steam_label_text,steam_positive_flag,steam_negative_flag,playtime_at_review_hours,playtime_stage,early_playtime_flag,recent_30d_flag,votes_up,weighted_vote_score,llm_sentiment,sentiment_score,llm_negative_or_mixed_flag,llm_positive_flag,llm_primary_issue,llm_urgency_candidate,high_urgency_flag,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation
0,success,189880229,619820,Heroes of Hammerwatch II,2025-03-10 15:54:38,2025-01-14,55,D31-D90,older_90d,negative,False,True,2.416667,1-5h,True,False,3,0.510774,negative,1,True,False,save_progression,high,True,최근 2일 동안 세이브 데이터가 두 번이나 삭제되어 게임을 플레이할 수 없음.,"세이브 데이터 손실 원인 파악 및 클라우드 저장 시스템 점검, 데이터 복구 방안 검토.",exact_match
1,success,213786980,619820,Heroes of Hammerwatch II,2025-12-21 03:28:01,2025-01-14,341,D181+,older_90d,positive,True,False,18.450000,5-20h,False,False,12,0.655379,positive,5,False,True,positive_praise,low,False,"초반에는 성장이 더뎌 지루할 수 있으나, 캐릭터와 마을을 육성할수록 매우 재미있어지는 게임. 싱글 플레이 경험에 만족함.","초반 진입 장벽을 낮추기 위한 튜토리얼 강화 또는 가이드 제공, 현재의 성장 루프 유지 및 강화.",exact_match
2,success,215844634,619820,Heroes of Hammerwatch II,2026-01-12 19:36:41,2025-01-14,363,D181+,older_90d,positive,True,False,30.783333,20-50h,False,False,0,0.500000,mixed,3,True,False,save_progression,high,True,"게임 자체는 매우 재미있으나, 리눅스로 OS를 변경하는 과정에서 세이브 데이터가 모두 소실되어 플레이를 중단함.",OS 변경 또는 PC 교체 시 세이브 데이터 호환성 및 클라우드 동기화 문제 조사 및 수정.,partial_match
3,success,185667334,619820,Heroes of Hammerwatch II,2025-01-15 20:08:55,2025-01-14,1,D0-D30,older_90d,positive,True,False,5.750000,5-20h,False,False,2,0.505464,positive,5,False,True,positive_praise,low,False,"그래픽과 클래스 시스템이 개선되었으며, 가격 대비 플레이 타임이 훌륭한 협동 게임으로 평가함.","현재의 그래픽 스타일과 유연한 클래스 시스템을 유지하고, 협동 플레이의 재미를 강화하는 업데이트를 지속할 것.",exact_match
4,success,186334804,619820,Heroes of Hammerwatch II,2025-01-25 00:44:23,2025-01-14,11,D0-D30,older_90d,negative,False,True,3.200000,1-5h,True,False,2,0.536785,negative,1,True,False,content_volume,high,True,"게임플레이가 반복적이고 콘텐츠가 부족하며, 현대적인 경쟁작들에 비해 전반적으로 수준이 낮다고 평가함.","적 종류 추가, 스킬 트리 확장, 게임플레이 다양성을 높일 수 있는 콘텐츠 업데이트 고려 필요.",exact_match


...

,priority_candidate,issue_count
0,하,10
1,중,6
2,상,1


최종 우선 검토 수준 분포:


,rule_priority_hint,issue_count
0,하,10
1,중,6
2,상,1


최종 '상' 제한 기준: 최대 5개
High urgency는 rule_priority_hint 계산에는 직접 사용하지 않고, priority_reason/llm_evidence_text에서 보조 지표로만 유지한다.

03-1번 여러 게임 실행 요약
batch log 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\03-1_postlaunch_preprocess_batch_log.csv


,game_key,appid,game_name,status,review_base_rows,issue_summary_rows,patch_ops_evidence_rows,tableau_source_rows,preprocess_dir,review_base_path,issue_summary_path,patch_ops_evidence_base_path,tableau_source_path,error_message
0,heroes_of_hammerwatch_2,619820,Heroes of Hammerwatch II,success,1000,20,20,1903,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\postl...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\postl...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\postl...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\postl...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\postl...,
1,necrosmith_2,2277320,Necrosmith 2,success,324,21,21,615,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\necrosmith_2\postlaunch_prepr...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\necrosmith_2\postlaunch_prepr...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\necrosmith_2\postlaunch_prepr...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\necrosmith_2\postlaunch_prepr...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\necrosmith_2\postlaunch_prepr...,
2,children_of_the_sun,1309950,Children of the Sun,success,883,19,19,1567,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\children_of_the_sun\postlaunc...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\children_of_the_sun\postlaunc...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\children_of_the_sun\postlaunc...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\children_of_the_sun\postlaunc...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\children_of_the_sun\postlaunc...,
3,laundry_store_simulator,3150440,Laundry Store Simulator,success,575,21,21,1064,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\laundry_store_simulator\postl...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\laundry_store_simulator\postl...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\laundry_store_simulator\postl...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\laundry_store_simulator\postl...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\laundry_store_simulator\postl...,
4,endoparasitic_2,2990640,Endoparasitic 2,success,245,17,17,477,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\postlaunch_pr...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\postlaunch_pr...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\postlaunch_pr...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\postlaunch_pr...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\postlaunch_pr...,


# 3. 게임별 산출물 통합 저장

`TARGET_GAME_KEYS`에 있는 게임별 실행 결과를 모아 `data/outputs/postlaunch/master` 폴더에 통합 저장한다.

이 셀은 위의 03-1 전처리 실행이 끝난 뒤 실행한다. 03번 LLM 원천 산출물과 03-1 전처리 산출물을 함께 master 폴더로 모으기 위한 단계이다.


In [4]:
# ============================================================
# 03-1 산출물 게임별 결과 통합
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

POSTLAUNCH_OUTPUT_DIR = ROOT / "data" / "outputs" / "postlaunch"
RUNS_DIR = POSTLAUNCH_OUTPUT_DIR / "runs"

MASTER_DIR = POSTLAUNCH_OUTPUT_DIR / "master"
MASTER_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PREPROCESS_DIR = MASTER_DIR / "postlaunch_preprocess_data"
MASTER_PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)

if not RUNS_DIR.exists():
    raise FileNotFoundError(f"게임별 runs 폴더가 없습니다: {RUNS_DIR}")


def read_run_csv(run_dir, relative_path):
    path = run_dir / relative_path

    if not path.exists():
        return None

    df = pd.read_csv(path, dtype={"recommendationid": "string"})
    df["source_run"] = run_dir.name
    return df


def concat_run_csv(relative_path, drop_duplicates_subset=None):
    dfs = []

    for run_dir in sorted(RUNS_DIR.iterdir()):
        if not run_dir.is_dir():
            continue

        df = read_run_csv(run_dir, relative_path)

        if df is not None and len(df) > 0:
            dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    merged = pd.concat(dfs, ignore_index=True)

    if drop_duplicates_subset:
        valid_subset = [col for col in drop_duplicates_subset if col in merged.columns]
        if valid_subset:
            merged = merged.drop_duplicates(subset=valid_subset, keep="last")

    return merged


# ============================================================
# 03번 LLM 원천 산출물 통합
# ============================================================

llm_input_reviews = concat_run_csv(
    "llm_input_reviews.csv",
    drop_duplicates_subset=["recommendationid"],
)

llm_filter_log = concat_run_csv(
    "llm_input_filter_log.csv",
)

llm_sample_summary = concat_run_csv(
    "llm_input_sample_summary.csv",
)

llm_review_result = concat_run_csv(
    "llm_review_analysis_result.csv",
    drop_duplicates_subset=["recommendationid"],
)

llm_issue_tags_flat = concat_run_csv(
    "llm_issue_tags_flat.csv",
    drop_duplicates_subset=["appid", "recommendationid", "llm_issue_category", "llm_issue_evidence"],
)

llm_input_reviews.to_csv(MASTER_DIR / "llm_input_reviews.csv", index=False, encoding="utf-8-sig")
llm_filter_log.to_csv(MASTER_DIR / "llm_input_filter_log.csv", index=False, encoding="utf-8-sig")
llm_sample_summary.to_csv(MASTER_DIR / "llm_input_sample_summary.csv", index=False, encoding="utf-8-sig")
llm_review_result.to_csv(MASTER_DIR / "llm_review_analysis_result.csv", index=False, encoding="utf-8-sig")
llm_issue_tags_flat.to_csv(MASTER_DIR / "llm_issue_tags_flat.csv", index=False, encoding="utf-8-sig")


# ============================================================
# 03-1번 게임별 전처리 산출물 통합
# ============================================================

postlaunch_review_base = concat_run_csv(
    "postlaunch_preprocess_data/postlaunch_review_base.csv",
    drop_duplicates_subset=["recommendationid"],
)

postlaunch_issue_summary = concat_run_csv(
    "postlaunch_preprocess_data/postlaunch_issue_summary.csv",
    drop_duplicates_subset=["appid", "llm_issue_category"],
)

postlaunch_patch_ops_evidence_base = concat_run_csv(
    "postlaunch_preprocess_data/postlaunch_patch_ops_evidence_base.csv",
    drop_duplicates_subset=["appid", "llm_issue_category"],
)

tableau_postlaunch_source = concat_run_csv(
    "postlaunch_preprocess_data/tableau_postlaunch_patch_ops_source.csv",
    drop_duplicates_subset=["appid", "recommendationid", "llm_issue_category"],
)

postlaunch_review_base.to_csv(
    MASTER_PREPROCESS_DIR / "postlaunch_review_base.csv",
    index=False,
    encoding="utf-8-sig",
)

postlaunch_issue_summary.to_csv(
    MASTER_PREPROCESS_DIR / "postlaunch_issue_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

postlaunch_patch_ops_evidence_base.to_csv(
    MASTER_PREPROCESS_DIR / "postlaunch_patch_ops_evidence_base.csv",
    index=False,
    encoding="utf-8-sig",
)

tableau_postlaunch_source.to_csv(
    MASTER_PREPROCESS_DIR / "tableau_postlaunch_patch_ops_source.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 통합 결과 확인
# ============================================================

check_rows = [
    {
        "파일": "llm_input_reviews.csv",
        "행 수": len(llm_input_reviews),
        "저장 위치": str(MASTER_DIR / "llm_input_reviews.csv"),
    },
    {
        "파일": "llm_review_analysis_result.csv",
        "행 수": len(llm_review_result),
        "저장 위치": str(MASTER_DIR / "llm_review_analysis_result.csv"),
    },
    {
        "파일": "llm_issue_tags_flat.csv",
        "행 수": len(llm_issue_tags_flat),
        "저장 위치": str(MASTER_DIR / "llm_issue_tags_flat.csv"),
    },
    {
        "파일": "postlaunch_review_base.csv",
        "행 수": len(postlaunch_review_base),
        "저장 위치": str(MASTER_PREPROCESS_DIR / "postlaunch_review_base.csv"),
    },
    {
        "파일": "postlaunch_issue_summary.csv",
        "행 수": len(postlaunch_issue_summary),
        "저장 위치": str(MASTER_PREPROCESS_DIR / "postlaunch_issue_summary.csv"),
    },
    {
        "파일": "postlaunch_patch_ops_evidence_base.csv",
        "행 수": len(postlaunch_patch_ops_evidence_base),
        "저장 위치": str(MASTER_PREPROCESS_DIR / "postlaunch_patch_ops_evidence_base.csv"),
    },
    {
        "파일": "tableau_postlaunch_patch_ops_source.csv",
        "행 수": len(tableau_postlaunch_source),
        "저장 위치": str(MASTER_PREPROCESS_DIR / "tableau_postlaunch_patch_ops_source.csv"),
    },
]

check_df = pd.DataFrame(check_rows)
display(check_df)

print("통합 완료")
print("통합된 게임 수:", postlaunch_review_base["appid"].nunique() if "appid" in postlaunch_review_base.columns else 0)

if "game_name" in postlaunch_review_base.columns:
    display(
        postlaunch_review_base
        .groupby(["appid", "game_name"], dropna=False)
        .agg(review_count=("recommendationid", "nunique"))
        .reset_index()
        .sort_values("review_count", ascending=False)
    )

,파일,행 수,저장 위치
0,llm_input_reviews.csv,3027,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\master\llm_input_reviews.csv
1,llm_review_analysis_result.csv,3027,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\master\llm_review_analysis_result.csv
2,llm_issue_tags_flat.csv,5729,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\master\llm_issue_tags_flat.csv
3,postlaunch_review_base.csv,3027,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\master\postlaunch_preprocess_data\...
4,postlaunch_issue_summary.csv,98,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\master\postlaunch_preprocess_data\...
5,postlaunch_patch_ops_evidence_base.csv,98,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\master\postlaunch_preprocess_data\...
6,tableau_postlaunch_patch_ops_source.csv,5626,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\master\postlaunch_preprocess_data\...


통합 완료
통합된 게임 수: 5


,appid,game_name,review_count
0,619820,Heroes of Hammerwatch II,1000
1,1309950,Children of the Sun,883
4,3150440,Laundry Store Simulator,575
2,2277320,Necrosmith 2,324
3,2990640,Endoparasitic 2,245


# 4. 산출 CSV 테이블 명세서

03-1에서는 03-2에서 실제로 사용할 산출물만 저장한다.

## 산출 파일 목록

| 파일명 | 역할 |
|---|---|
| `postlaunch_review_base.csv` | 리뷰 1개 단위 전처리 결과 |
| `postlaunch_issue_summary.csv` | 이슈별 반복성, 부정·혼합, Steam 비추천, 최근성, 플레이타임 근거 요약 |
| `postlaunch_patch_ops_evidence_base.csv` | 03-2 LLM 패치·운영 전략 생성용 근거 데이터 |
| `tableau_postlaunch_patch_ops_source.csv` | Tableau/보고서용 원천 데이터 |

추가로 게임별 runs 결과를 통합하여 아래 위치에도 저장한다.

| 저장 위치 | 파일명 | 역할 |
|---|---|---|
| `data/outputs/postlaunch/master/` | `llm_input_reviews.csv` | 03번 LLM 입력 리뷰 통합본 |
| `data/outputs/postlaunch/master/` | `llm_review_analysis_result.csv` | 03번 리뷰 단위 LLM 결과 통합본 |
| `data/outputs/postlaunch/master/` | `llm_issue_tags_flat.csv` | 03번 이슈 태그 단위 LLM 결과 통합본 |
| `data/outputs/postlaunch/master/postlaunch_preprocess_data/` | `postlaunch_review_base.csv` | 03-1 리뷰 단위 전처리 결과 통합본 |
| `data/outputs/postlaunch/master/postlaunch_preprocess_data/` | `postlaunch_issue_summary.csv` | 03-1 이슈별 근거 요약 통합본 |
| `data/outputs/postlaunch/master/postlaunch_preprocess_data/` | `postlaunch_patch_ops_evidence_base.csv` | 03-2 입력용 근거 데이터 통합본 |
| `data/outputs/postlaunch/master/postlaunch_preprocess_data/` | `tableau_postlaunch_patch_ops_source.csv` | Tableau/보고서용 원천 데이터 통합본 |

---

## postlaunch_review_base.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `recommendationid` | Steam 리뷰 고유 ID | `221779395` |
| `appid` | Steam 게임 고유 ID | `1466060` |
| `game_name` | 게임명 | `Tainted Grail: The Fall of Avalon` |
| `review_datetime` | 리뷰 작성 일시 | `2026-04-05 16:40:12` |
| `review_recency_group` | 분석 데이터 내 최신 리뷰일 기준 최근성 구간 | `last_30d` |
| `steam_label_text` | Steam 추천/비추천 라벨 | `positive` |
| `playtime_at_review_hours` | 리뷰 작성 시점 플레이타임 | `12.5` |
| `playtime_stage` | 리뷰 작성 시점 플레이타임 구간 | `5-20h` |
| `llm_sentiment` | LLM이 리뷰 내용을 보고 분류한 감정 | `mixed` |
| `llm_urgency_candidate` | LLM이 리뷰 내용을 보고 분류한 시급도 후보 | `high` |
| `high_urgency_flag` | High urgency 여부. 우선 검토 수준 계산 기준이 아니라 보조 참고 지표 | `True` |
| `llm_review_summary` | LLM이 작성한 리뷰 요약 | `전투와 탐험은 좋지만 버그를 지적함` |
| `llm_suggested_action` | LLM이 리뷰 내용을 바탕으로 정리한 개선 방향 후보 | `진행 방해 버그를 우선 확인` |

---

## postlaunch_issue_summary.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `llm_issue_category` | LLM이 추출한 이슈 카테고리 | `gameplay_loop` |
| `issue_name_kor` | 이슈 한글명 | `게임플레이 루프` |
| `affected_review_count` | 해당 이슈가 언급된 리뷰 수 | `239` |
| `negative_mixed_review_count` |  LLM이 이슈 태그 단위에서 부정 또는 혼합 맥락으로 분류한 리뷰 수. Steam 비추천 라벨과는 별개의 LLM 분류값 | `130` |
| `steam_negative_review_count` | Steam 비추천 리뷰 중 해당 이슈가 언급된 리뷰 수 | `64` |
| `high_urgency_review_count` | LLM이 High urgency 후보로 분류한 리뷰 수. 우선 검토 수준 계산에는 직접 사용하지 않음 | `42` |
| `high_urgency_rate` | 영향 리뷰 중 High urgency 후보 비율. 보조 참고 지표 | `0.176` |
| `negative_mixed_rate` | 영향 리뷰 중 부정/혼합 비율 | `0.544` |
| `steam_negative_rate` | 영향 리뷰 중 Steam 비추천 비율 | `0.269` |
| `recent_30d_negative_mixed_review_count` | 분석 데이터 내 최신 리뷰일 기준 최근 30일 부정/혼합 리뷰 수 | `12` |
| `early_playtime_negative_mixed_review_count` | 0~5시간 구간 부정/혼합 리뷰 수 | `8` |
| `action_group_hint` | 이슈 성격과 집계값 기반 대응 구분 | `단기 개선` |
| `rule_priority_hint` | 규칙 기반 우선 검토 수준 | `상` |
| `priority_rule_detail` | 우선 검토 수준이 부여된 규칙 설명 | `부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복` |
| `priority_reason` | 우선 검토 힌트의 근거 요약 | `영향 리뷰 239개, 부정·혼합 130개...` |
| `patch_ops_note` | 패치·운영 해석 메모 | `반복 피로, 목표 구조, 보상 흐름을 점검...` |

---

## postlaunch_patch_ops_evidence_base.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `llm_issue_category` | LLM이 추출한 이슈 카테고리 | `save_progression` |
| `issue_name_kor` | 이슈 한글명 | `저장/진행` |
| `action_group_hint` | 03-1에서 계산한 대응 구분 | `즉시 확인` |
| `rule_priority_hint` | 03-1에서 계산한 고정 우선 검토 수준 | `상` |
| `priority_rule_detail` | 우선 검토 수준이 부여된 규칙 설명 | `플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락` |
| `priority_reason` | 집계 기반 근거 문장 | `영향 리뷰 26개, 부정·혼합 23개...` |
| `patch_ops_note` | 이슈별 기본 대응 메모 | `저장 손실, 진행 막힘...` |
| `llm_evidence_text` | 03-2 LLM 프롬프트에 넣을 근거 블록 | `[ISSUE] ... [/ISSUE]` |

---

## tableau_postlaunch_patch_ops_source.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `review_datetime` | 리뷰 작성 일시 | `2026-04-05 16:40:12` |
| `review_recency_group` | 최근성 구간 | `last_30d` |
| `playtime_stage` | 플레이타임 구간 | `20-50h` |
| `llm_issue_category` | 이슈 카테고리 | `performance` |
| `issue_name_kor` | 이슈 한글명 | `성능` |
| `llm_issue_sentiment` | 해당 이슈가 리뷰에서 나타난 감정 | `negative` |
| `issue_negative_or_mixed_flag` | 부정/혼합 맥락 여부 | `True` |
| `action_group_hint` | 대응 구분 힌트 | `즉시 확인` |
| `rule_priority_hint` | 우선 검토 힌트 | `상` |
| `priority_rule_detail` | 우선 검토 힌트 부여 규칙 | `부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복` |
| `high_urgency_rate` | 이슈별 High urgency 후보 비율. 보조 참고 지표 | `0.42` |
| `patch_ops_note` | 패치·운영 해석 메모 | `성능 저하와 프레임 드랍을 우선 확인...` |
